## Pipeline vs Manual Inference


In [ ]:
pipeline() vs Manual Inference

Let's understand the same task in two ways: first using pipeline(), then manually.

We'll use sentiment analysis as the example.

Step 1 — Concept
Using pipeline()

pipeline() is a high-level API in Hugging Face Transformers that combines multiple steps for you.

pipeline()
   ↓
Tokenizer
   ↓
Model
   ↓
Post-processing
   ↓
Final Result

You only provide the text.

Manual approach

When doing it manually, you write each step yourself:

Text
 ↓
Tokenizer
 ↓
Model
 ↓
Raw logits
 ↓
Post-processing
 ↓
Label + Score

This gives you more control and helps you understand what happens internally.

In [ ]:
Step 2 — Simple Example

Suppose we give:

"I love this product!"

The model should return something like:

POSITIVE

with a confidence score.

Pipeline
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("I love this product!")

print(result)

Output:

[{'label': 'POSITIVE', 'score': 0.9998}]

The pipeline automatically handles:

Loading tokenizer
Loading model
Tokenizing text
Running inference
Converting model output into label + score

In [ ]:
## Step 3 — Perform the Same Task Manually

import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 3. Input text
text = "I love this product!"

# 4. Tokenization
inputs = tokenizer(text, return_tensors="pt")

# 5. Model inference
with torch.no_grad():
    outputs = model(**inputs)

# 6. Get logits
logits = outputs.logits

# 7. Convert logits to probabilities
probabilities = torch.softmax(logits, dim=-1)

# 8. Find highest probability
predicted_class = torch.argmax(probabilities, dim=-1).item()

# 9. Convert class ID to label
label = model.config.id2label[predicted_class]

# 10. Get confidence score
score = probabilities[0][predicted_class].item()

print("Label:", label)
print("Score:", score)

In [ ]:
Step 4 — Understand Each Step
1. Tokenizer
inputs = tokenizer(text, return_tensors="pt")

Converts:

"I love this product!"

into numbers the model understands.

Conceptually:

Text
 ↓
Tokens
 ↓
Token IDs
 ↓
Attention Mask

Example:

"I love AI"

↓
["i", "love", "ai"]

↓
[1045, 2293, 9938]

The actual IDs depend on the tokenizer

In [ ]:
2. Model
outputs = model(**inputs)

The model receives the token IDs.

Token IDs
    ↓
Transformer
    ↓
Neural network layers
    ↓
Logits

The model doesn't directly return "POSITIVE".

It initially returns logits.

Example:

NEGATIVE → -3.2
POSITIVE →  4.8

In [ ]:
3. Post-processing

We convert logits into probabilities:

probabilities = torch.softmax(logits, dim=-1)

Example:

NEGATIVE → 0.0003
POSITIVE → 0.9997

Then find the highest probability:

predicted_class = torch.argmax(probabilities, dim=-1)

Then convert the class ID into a readable label:

label = model.config.id2label[predicted_class]

Result:

POSITIVE

| Step                | `pipeline()` | Manual    |
| ------------------- | ------------ | --------- |
| Load tokenizer      | Automatic    | You do it |
| Load model          | Automatic    | You do it |
| Tokenization        | Automatic    | You do it |
| Model inference     | Automatic    | You do it |
| Logits handling     | Automatic    | You do it |
| Softmax/probability | Automatic    | You do it |
| Label conversion    | Automatic    | You do it |
| Control             | Less         | More      |
| Code                | Very short   | More code |
| Learning internals  | Less         | Better    |


In [ ]:
Step 6 — Experiment

Change:

text = "I love this product!"

to:

text = "I hate this product!"

The result should change from approximately:

POSITIVE → high probability

to:

NEGATIVE → high probability

A Hugging Face pipeline is a high-level API that combines tokenization, model inference, and post-processing into a simple interface. In manual inference, we explicitly load the tokenizer and model, tokenize the input, pass it to the model, process the logits, and convert them into the final prediction.


In [ ]:
pipeline() = all steps packaged together
Manual = we control each step individually.

In [ ]:
This is a mandatory and important notebook.
First perform inference using:

pipeline()
Then perform the same task manually.
Understand the difference between:
Pipeline
   ↓
Tokenizer
   ↓
Model
   ↓
Post-processing

and manually implementing:
Input
 ↓
Tokenizer
 ↓
Model
 ↓
Model Output
 ↓
Post-processing
 ↓
Final Result
Explain:
What the pipeline abstracts.
What happens behind the pipeline.
Advantages of pipelines.
Limitations of pipelines.
When manual model loading is useful.

In [3]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("I love this product!")

print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998855590820312}]


In [2]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Step 1: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Step 3: Input
text = "I love this product!"

# Step 4: Tokenization
inputs = tokenizer(text, return_tensors="pt")

# Step 5: Model inference
with torch.no_grad():
    outputs = model(**inputs)

# Step 6: Get logits
logits = outputs.logits

# Step 7: Convert logits to probabilities
probabilities = torch.softmax(logits, dim=-1)

# Step 8: Find predicted class
predicted_class = torch.argmax(probabilities, dim=-1).item()

# Step 9: Convert class ID to label
label = model.config.id2label[predicted_class]

# Step 10: Get confidence score
score = probabilities[0][predicted_class].item()

print("Label:", label)
print("Score:", score)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Label: POSITIVE
Score: 0.9998855590820312


In [ ]:
Hugging Face pipeline() — What Happens Behind It
1. What does pipeline() abstract?

Abstraction means pipeline() hides the detailed steps and gives you a simple interface.

Without pipeline:

Text
 ↓
Tokenizer
 ↓
Model
 ↓
Logits
 ↓
Post-processing
 ↓
Result

With pipeline:

Text
 ↓
pipeline()
 ↓
Result

Hugging Face pipelines provide a high-level interface that abstracts preprocessing, tokenization, model inference, and post-processing. They are useful for quickly building and testing AI applications with minimal code. However, pipelines provide less control over model inputs, outputs, intermediate representations, and optimization. Manual model loading is useful when we need custom preprocessing, access to logits or hidden states, custom post-processing, multiple models, or production-level performance optimization.

| Feature                 | `pipeline()`      | Manual Loading |
| ----------------------- | ----------------- | -------------- |
| Code                    | Very little       | More code      |
| Ease of use             | ⭐⭐⭐⭐⭐             | ⭐⭐⭐            |
| Control                 | Low               | High           |
| Customization           | Limited           | High           |
| Raw logits              | Less direct       | Easy           |
| Hidden states           | Less direct       | Easy           |
| Custom preprocessing    | Limited           | Full control   |
| Custom post-processing  | Limited           | Full control   |
| Prototyping             | Excellent         | Good           |
| Learning internals      | Basic             | Excellent      |
| Complex applications    | Sometimes limited | Better         |
| Production optimization | Less control      | More control   |
